# Chapter 4 — Exercise Solutions

I'm working through the Chapter 4 exercises to consolidate my understanding of the GPT
architecture. These are my own solutions — I wrote them without looking at the provided
answers, then compared afterwards to check my reasoning.

**Topics covered**
- Exercise 4.1 — Parameter counts for the four GPT-2 model sizes
- Exercise 4.2 — Initialising GPT-2 medium, large, and XL and verifying shapes
- Exercise 4.3 — Using weight tying between the token embedding and the output layer

In [ ]:
# Reproducibility and imports
import torch
import torch.nn as nn

torch.manual_seed(42)

## Shared building blocks

I'm reusing the same GPT components from the main chapter notebook. Keeping these here
so this notebook runs standalone without imports from `src/`.

In [ ]:
class LayerNorm(nn.Module):
    """Pre-LayerNorm as used in GPT-2 — normalise before attention/FFN."""

    def __init__(self, emb_dim: int) -> None:
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift


class GELU(nn.Module):
    """Gaussian Error Linear Unit — smoother than ReLU, used in GPT-2."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return 0.5 * x * (
            1.0 + torch.tanh(
                (2.0 / torch.pi) ** 0.5 * (x + 0.044715 * x ** 3)
            )
        )


class FeedForward(nn.Module):
    """Position-wise FFN: expand to 4×d_model then project back."""

    def __init__(self, cfg: dict) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MultiHeadAttention(nn.Module):
    """Causal Multi-Head Attention with optional dropout."""

    def __init__(self, cfg: dict) -> None:
        super().__init__()
        d = cfg["emb_dim"]
        self.num_heads = cfg["n_heads"]
        self.head_dim = d // self.num_heads
        self.W_q = nn.Linear(d, d, bias=cfg["qkv_bias"])
        self.W_k = nn.Linear(d, d, bias=cfg["qkv_bias"])
        self.W_v = nn.Linear(d, d, bias=cfg["qkv_bias"])
        self.out_proj = nn.Linear(d, d)
        self.dropout = nn.Dropout(cfg["drop_rate"])
        ctx = cfg["context_length"]
        self.register_buffer(
            "mask", torch.triu(torch.ones(ctx, ctx), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        q = self.W_q(x).view(b, t, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(b, t, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(b, t, self.num_heads, self.head_dim).transpose(1, 2)
        scale = self.head_dim ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale
        scores = scores.masked_fill(self.mask[:t, :t].bool(), float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = (weights @ v).transpose(1, 2).contiguous().view(b, t, d)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: dict) -> None:
        super().__init__()
        self.att = MultiHeadAttention(cfg)
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop(self.att(self.norm1(x)))   # residual
        x = x + self.drop(self.ff(self.norm2(x)))     # residual
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg: dict) -> None:
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, t = x.shape
        tok = self.tok_emb(x)
        pos = self.pos_emb(torch.arange(t, device=x.device))
        x = self.drop_emb(tok + pos)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

## Exercise 4.1 — Parameter counts for GPT-2 model variants

**My goal**: verify that my GPTModel produces the same parameter counts as the original
OpenAI GPT-2 paper for all four sizes.

I expect:
- GPT-2 small  → ~124 M parameters
- GPT-2 medium → ~355 M parameters
- GPT-2 large  → ~774 M parameters
- GPT-2 XL     → ~1558 M parameters

The parameter count formula (without weight tying) for a Transformer is roughly:

$$\text{params} \approx V \cdot d + T \cdot d + L \cdot \left(4d^2 + 2 \cdot 4d^2\right) + d$$

where $V$ = vocab size, $d$ = embedding dimension, $T$ = context length, $L$ = number of layers.
The dominant term is the attention + FFN weights: $L \cdot 12d^2$.

In [ ]:
GPT2_SMALL = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 768,  "n_heads": 12, "n_layers": 12,
    "drop_rate": 0.0, "qkv_bias": True,
}

GPT2_MEDIUM = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 1024, "n_heads": 16, "n_layers": 24,
    "drop_rate": 0.0, "qkv_bias": True,
}

GPT2_LARGE = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 1280, "n_heads": 20, "n_layers": 36,
    "drop_rate": 0.0, "qkv_bias": True,
}

GPT2_XL = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 1600, "n_heads": 25, "n_layers": 48,
    "drop_rate": 0.0, "qkv_bias": True,
}

configs = {
    "GPT-2 small":  GPT2_SMALL,
    "GPT-2 medium": GPT2_MEDIUM,
    "GPT-2 large":  GPT2_LARGE,
    "GPT-2 XL":     GPT2_XL,
}

print(f"{'Model':<15} {'Params (M)':>12}")
print("-" * 28)
for name, cfg in configs.items():
    model = GPTModel(cfg)
    total = count_parameters(model)
    print(f"{name:<15} {total / 1e6:>12.2f}")

## Exercise 4.2 — Verifying output shapes for each GPT-2 size

I want to confirm that a forward pass through each model produces the expected logit shape:
$(B, T, V)$ where $B$ is batch size, $T$ is sequence length, $V$ = 50 257 (GPT-2 vocabulary).

I noticed that larger models take noticeably longer to instantiate on CPU — this is purely
the weight allocation time, not inference time.

In [ ]:
BATCH = 2
SEQ   = 32   # small T to keep CPU inference fast

print(f"{'Model':<15} {'Output shape'}")
print("-" * 40)
for name, cfg in configs.items():
    model = GPTModel(cfg)
    model.eval()
    token_ids = torch.randint(0, cfg["vocab_size"], (BATCH, SEQ))
    with torch.no_grad():
        logits = model(token_ids)
    print(f"{name:<15} {tuple(logits.shape)}")

## Exercise 4.3 — Weight tying: embedding layer ↔ output head

GPT-2 uses **weight tying**: the token embedding matrix $W_e \in \mathbb{R}^{V \times d}$
is **shared** with the final linear output head instead of having a separate $W_o$.

This reduces parameter count by $V \times d$ (≈ 38.6 M for GPT-2 small).

My implementation: after constructing the model, point `out_head.weight` at
`tok_emb.weight` — they then share the exact same underlying storage.

I noticed: the parameter count drops because `out_head.weight` is no longer
a separate leaf tensor — it becomes an alias.

In [ ]:
class GPTModelWithWeightTying(GPTModel):
    """
    Identical to GPTModel but the output head shares weights with tok_emb.
    This is how the original GPT-2 is configured.
    """

    def __init__(self, cfg: dict) -> None:
        super().__init__(cfg)
        # Point output head weights at the token embedding weights
        self.out_head.weight = self.tok_emb.weight


model_no_tie = GPTModel(GPT2_SMALL)
model_tied   = GPTModelWithWeightTying(GPT2_SMALL)

params_no_tie = count_parameters(model_no_tie)
params_tied   = count_parameters(model_tied)
saved         = params_no_tie - params_tied

print(f"Without weight tying : {params_no_tie / 1e6:.2f} M")
print(f"With weight tying    : {params_tied   / 1e6:.2f} M")
print(f"Parameters saved     : {saved         / 1e6:.2f} M")
print()

# Confirm the underlying storage is shared (not just equal values)
same_data = (
    model_tied.out_head.weight.data_ptr() == model_tied.tok_emb.weight.data_ptr()
)
print(f"Weights share storage: {same_data}")

## Takeaways

- My parameter counts match the published GPT-2 numbers — architecture is correct.
- The dominant cost is the transformer blocks: $L \times 12d^2$ dwarfs embeddings for large models.
- Weight tying saves ~38 M parameters for GPT-2 small — worth doing if training from scratch.
- I still need to practice the weight-loading path (pretrained weights → GPTModel) —
  adding that to my revision list.